In [1]:
!rm -rf /content/mimir-v0/
!git clone https://github.com/AdityaSinghDevs/mimir-v0

Cloning into 'mimir-v0'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 180 (delta 65), reused 147 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 60.58 KiB | 10.10 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [2]:
%cd mimir-v0

/content/mimir-v0


In [3]:
!pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement pywin32==311 (from versions: none)
ERROR: No matching distribution found for pywin32==311


In [4]:
from configs.config_loader import load_configs
from core.model_loader import load_model_and_tokenizer
from core.incident_loader import get_incident_ids, load_incident
from core.prompt_builder import prompt_builder
from core.generator import chat_builder, generate_response
from utils.save import save_results

In [5]:
base_cfg = load_configs("base_configs")
model_cfg = load_configs("model_configs")

In [6]:
model, tokenizer = load_model_and_tokenizer(
    model_name = model_cfg["model"]["name"],
    torch_dtype = model_cfg["model"]["torch_dtype"],
    device_map = model_cfg["model"]["device_map"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# import importlib
# import core.generator

# importlib.reload(core.generator)

In [8]:
# from core.generator import chat_builder, generate_response

In [13]:
incident_ids = get_incident_ids()
num_runs = base_cfg["experiment"]["n_runs"]
prompt_type = "structured"

for id in incident_ids:

    incident = load_incident(incident_id=id)
    messages = prompt_builder(prompt_type=prompt_type,
                              incident=incident)

    all_run_results = []

    for run_id in range(num_runs):

        inputs = chat_builder(model=model,
                               tokenizer=tokenizer,
                               messages=messages)

        response = generate_response(
            inputs=inputs,
            model=model,
            tokenizer=tokenizer,
            do_sample = base_cfg["experiment"]["do_sample"],
            max_tokens= base_cfg["experiment"]["max_tokens"],
            temperature = base_cfg["experiment"]["temperature"],
            top_p = base_cfg["experiment"]["top_p"],
        )

        run_result = {
            "run_id": run_id + 1,
            "response": response,
            }

        all_run_results.append(run_result)


    result = {
        "incident_id": incident["id"],
        "prompt_type": prompt_type,
        "model": model_cfg["model"]["name"],
        "temperature": base_cfg["experiment"]["temperature"],
        "top_p": base_cfg["experiment"]["top_p"],
        "max_tokens": base_cfg["experiment"]["max_tokens"],
        "do_sample": base_cfg["experiment"]["do_sample"],
        "responses": all_run_results
    }

    print(type(result))

    save_results(
        incident_id = f"incident_{id}",
        results = result,
        prompt_type = prompt_type,
        rag= "no_rag"

    )

<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>


In [10]:
file_path = "/content/mimir-v0/results/raw/incident_incident_01_freeform_no_rag.json"

with open(file_path, "r") as file:
    content = file.read()

print(content)

{
  "incident_id": "incident_01",
  "prompt_type": "freeform",
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "temperature": 0.7,
  "top_p": 0.9,
  "max_tokens": 150,
  "do_sample": true,
  "responses": [
    {
      "run_id": 1,
      "response": "Root Cause :\nUpstream service timeout leading to degraded responses\n\nSupporting Evidence :\n- 2026-02-22T14:21:12.266Z gateway-api-6f4b9d7c8d-mp9k2 stdout {\"ts\":\"2026-02-22T14:21:12.266Z\",\"level\":\"ERROR\",\"component\":\"http-client\",\"msg\":\"timeout contacting dependency\",\"request_id\":\"c0e9f1aa\",\"timeout_ms\":5000}"
    },
    {
      "run_id": 2,
      "response": "Root Cause :\nHigh-latency connection issue leading to 504 Gateway Timeout responses between services.\n\nSupporting Evidence :\n- 2026-02-22T14:21:03.112Z: ingress-nginx logs show a 504 error with a 5.002-second upstream response time.\n- 2026-02-22T14:21:04.087Z: ingress-nginx logs warn about an upstream timeout.\n- 2026-02-22T14:21:10.118Z: ingress-nginx logs anot

In [14]:
import shutil
from google.colab import files

folder_path = "/content/mimir-v0/results/raw"
zip_path = "/content/results-raw.zip"

shutil.make_archive(zip_path.replace(".zip",""), 'zip', folder_path)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>